In [0]:
%sql
CREATE CATALOG IF NOT EXISTS hpn
    MANAGED LOCATION 'abfss://managed@adlshpn.dfs.core.windows.net/';
COMMENT ON CATALOG hpn IS 'HPN Data Lake - Catálogo Gerenciado (medallion via schemas)';

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS hpn.1_bronze COMMENT "Dados Brutos ingeridos da landing";
CREATE SCHEMA IF NOT EXISTS hpn.2_silver COMMENT "Dados limpos e conformados";
CREATE SCHEMA IF NOT EXISTS hpn.3_gold COMMENT "Dados prontos para consumo BI/Genie";

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS hpn.1_bronze.volume;
CREATE VOLUME IF NOT EXISTS hpn.1_bronze.ops;

In [0]:
%sql
--DROP VOLUME IF EXISTS hpn.1_bronze.volume

In [0]:
src = "abfss://landing@adlshpn.dfs.core.windows.net/customer"

# Checkpoint — onde o Auto Loader anota "quais arquivos eu já processei". É a memória dele. Se você rodar de novo, ele lê aqui e pula o que já entrou (por isso é incremental). Está num Volume do UC.
chk = "/Volumes/hpn/1_bronze/ops/customer_chk"

#Schema location — o Auto Loader infere o schema dos Parquet e guarda aqui. Assim ele detecta se uma coluna nova aparecer no futuro (schema evolution).
sch = "/Volumes/hpn/1_bronze/ops/customer_schema"

# df Aqui é um DataFrame de streaming — ainda não leu nada. Spark é lazy: só descreve o "de onde e como ler". Nada acontece até ter um destino de escrita.

df = (spark.readStream.format("cloudFiles")     # Lê em modo streaming (incremental). Spark.Read lê tudo de uma vez
      .option("cloudFiles.format", "parquet")   # Informa o tipo de arquivo para o auto loader
      .option("cloudFiles.schemaLocation",sch)  # Onde guardar/ler schema inferido pelo auto loader
      .load(src)                                # Aponta para pasta de origem e monta o dataframe de streaming 
      )

# Esse é bloco de escrita
(df.writeStream
 .option("checkpointLocation", chk)             # onde guardar o checkpoint/progresso. Obrigatório em streaming. Garante incremental e tolerância à falha
 .option("mergeSchema", "true")                 # schema evolution
 .outputMode("append")                          #
 .trigger(availableNow=True)                    # o modo de disparo. "processa tudo que já chegou agora, e depois PARA". É streaming rodando como se fosse batch — perfeito pra carga sob demanda. (Alternativas: processingTime="1 minute" roda a cada minuto sem parar; sem trigger, roda contínuo pra sempre.)
 .toTable("hpn.1_bronze.customer")              # grava numa tabela gerenciada do UC.
 )

In [0]:
# 1. lista os secrets do scope (só nomes, sem valor) — confirma que o scope enxerga o cofre
dbutils.secrets.list("hpn-db")

# 2. tenta ler um secret — confirma acesso de runtime
host = dbutils.secrets.get(scope="hpn-db", key="kv-postgres-host")
print("Leu o host? tamanho =", len(host))   # imprime o tamanho; o valor sai [REDACTED] por segurança
